# Import 

In [ ]:
# WEEK 6
# DIABETES DATASET:
# Xây dựng 1 classifier vừa dự đoán xem 1 người có bị béo phì (Obesity) và có bị tiểu đường (Diabetic) hay không
# Sẽ có 4 kết quả: Bị cả 2 bệnh, Chỉ bị tiểu đường, Chỉ bị béo phì, Khỏe mạnh

# Import libraries:
!pip install lazypredict
import lazypredict
from lazypredict.Supervised import LazyClassifier
from ydata_profiling import ProfileReport
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report
from sklearn.svm import SVC
from sklearn.multioutput import MultiOutputClassifier

# Import dataset:
diabetes_df = pd.read_csv(r'/Users/ngocta/Desktop/CoderSchool-AI/Week6_7/diabetes_data.csv', sep = ',')
target = ['Obesity', 'DiabeticClass']

# Checking data statistics:
# diabetes_profile = ProfileReport(diabetes_df, title = "Diabetes Report", explorative = True)
# diabetes_profile.to_file("Diabetes Report.html")

# Separate target columns from the dataset:
x = diabetes_df.drop(labels = target, axis = 1)
y = diabetes_df[target]
y_1st_target = diabetes_df['Obesity']

# FIND THE OPTIMAL MODEL FOR 1ST TARGET (OBESITY):
# Split to train & test datasets:
x_train, x_test, y_1st_target_train, y_1st_target_test = train_test_split(x,y_1st_target, test_size = 0.2, random_state = 100)

# Create pipleline to handle boolean data:
boolean_cols = ['Gender','ExcessUrination', 'Polydipsia', 'WeightLossSudden',
       'Fatigue', 'Polyphagia', 'GenitalThrush', 'BlurredVision', 'Itching',
       'Irritability', 'DelayHealing', 'PartialPsoriasis', 'MuscleStiffness',
       'Alopecia']

ord_transformer = Pipeline(steps = [
    ('imputer', SimpleImputer(strategy = 'most_frequent')),
    ('encoder', OrdinalEncoder(categories = [['Male', 'Female']] + [['No','Yes']]*(len(boolean_cols)-1)))
])

# Checking the output:
#output = ord_transformer.fit_transform(x_train[boolean_cols])
#for i, j in zip(x_train[boolean_cols].values, output):
    #print("Before: {}. After: {}".format(i,j))

# Transform boolean text columns:
preprocessor = ColumnTransformer(transformers = [
    ('Ordinal features', ord_transformer, boolean_cols)
])

# Find the optimal model:
x_train_transformed = preprocessor.fit_transform(x_train)
x_test_transformed = preprocessor.transform(x_test)

clf = LazyClassifier(verbose = 0, ignore_warnings = True, custom_metric = None)
models, predictions = clf.fit(x_train_transformed, x_test_transformed, y_1st_target_train, y_1st_target_test)
print(predictions)


In [ ]:
# WRAP THE BASE MODEL (SVM) WITH MUTLIOUTPUT CLASSIFIER TO HANDLE MULTIPLE TARGETS (OBESITY & DIABETES CLASS)
training_model = MultiOutputClassifier(SVC())

# Full dataset split to train & test set and ensure even distribution of all the class in each columns:
stratified_cols = y['Obesity'] + "-" + ['DiabeticClass']

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size = 0.2, random_state = 100, stratify = stratified_cols
)

# Build full pipeline
model = Pipeline(steps=[
    ('Preprocessor', preprocessor),
    ('model', training_model)
])

# Train the multi-target model
model.fit(x_train, y_train)

# Predict and evaluate
y_predict = model.predict(x_test)
y_predict_df = pd.DataFrame(y_predict, columns=['Obesity', 'DiabeticClass'])

# Combine Obesity and Diabetes classification results into a final prediction:
def get_analysis(obesity_result, diabetes_result):
    if obesity_result == 'Yes' and diabetes_result == 'Positive':
        return "Bị cả 2 bệnh"
    elif obesity_result == 'No' and diabetes_result == 'Positive':
        return "Chỉ bị tiểu đường"
    elif obesity_result == 'Yes' and diabetes_result == 'Negative':
        return "Chỉ bị béo phì"
    else:
        return "Khỏe mạnh"

get_analysis_train = [get_analysis(a, b) for a, b in zip(y_test['Obesity'], y_test['DiabeticClass'])]
get_analysis_test = [get_analysis(x, y) for x, y in zip(y_predict_df['Obesity'], y_predict_df['DiabeticClass'])]

print("OBESITY REPORT:")
print(classification_report(y_test['Obesity'], y_predict_df['Obesity']))

print("\n DIABETES REPORT:")
print(classification_report(y_test['DiabeticClass'], y_predict_df['DiabeticClass']))

print("\n COMBINED FINAL REPORT:")
print(classification_report(get_analysis_train, get_analysis_test))

In [ ]:
# STROKE DATASET:
# Xây dựng 1 classifier vừa dự đoán xem 1 người có bị đột quỵ (stroke) hay không

# Import libraries:
import pandas as pd
import lazypredict
from ydata_profiling import ProfileReport
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from lazypredict.Supervised import LazyClassifier
from sklearn.neighbors import NearestCentroid
from sklearn.metrics import classification_report

# Import dataset:
stroke_df = pd.read_csv(r'/Users/ngocta/Desktop/CoderSchool-AI/Week6_7/stroke_classification.csv', sep = ',')
target_2 = 'stroke'

# Review data statistics:
#stroke_profile = ProfileReport(stroke_df, title = 'Stroke Report', explorative = True)
#stroke_profile.to_file('Stroke Report.html')

# Remove row with "Other" value in Gender Column (Only 1 value of "Other" in the whole column):
stroke_df  = stroke_df[stroke_df['gender'] != "Other"]

# Separate target column from the dataset:
x_2 = stroke_df.drop(labels = [target_2, 'pat_id'], axis = 1)
y_2 = stroke_df[target_2]

# Split to train & test sets:
x_train_2, x_test_2, y_train_2, y_test_2 = train_test_split(x_2, y_2, test_size = 0.2, 
                                                            random_state = 100, stratify = y_2)

# Handle boolean columns:
bool_cols = ['gender', 'hypertension', 'heart_disease',
       'work_related_stress', 'urban_residence', 'smokes']

ord_transformer_2 = Pipeline(steps = [
    ('imputer', SimpleImputer(strategy = 'most_frequent')),
    ('encoder', OrdinalEncoder(categories = [['Male', 'Female']] + [[0,1]]*(len(bool_cols)-1)))
])

# Check the output of ordinal transformer:
# ord_output = ord_transformer_2.fit_transform(x_train_2[bool_cols])
#for i, j in zip(x_train_2[bool_cols].values, ord_output):
    #print("Before: {}. After: {}".format(i, j))


# Handle numerical columns:
num_cols = ['age', 'avg_glucose_level', 'bmi']

num_transformer = Pipeline(steps = [
    ('imputer', SimpleImputer(strategy = 'mean')),
    ('scaler', StandardScaler())
])

# Check the output of numerical transformer:
#num_output = num_transformer.fit_transform(x_train_2[num_cols])
#for i, j in zip(x_train_2[num_cols].values, num_output):
    #print("Before {}. After {}".format(i, j))

# Transform columns:
preprocessor = ColumnTransformer(transformers = [
    ('Ordinal features', ord_transformer_2, bool_cols),
    ('Numerical features', num_transformer, num_cols)
])

# Find the optimal model:
x_train_2b = preprocessor.fit_transform(x_train_2)
x_test_2b = preprocessor.transform(x_test_2)
clf = LazyClassifier(verbose = 0, ignore_warnings = True, custom_metric = None)
models_2, predictions_2 = clf.fit(x_train_2b, x_test_2b, y_train_2, y_test_2)
print(predictions_2)

In [ ]:
# Optimize the model:
params = {
    "metric": ['euclidean', 'manhattan']
}
                
optimized_model = GridSearchCV(
    estimator = NearestCentroid(),
    param_grid = params,
    scoring = 'recall',
    verbose = 0
)

# Build the full pipelines:
model_2 = Pipeline(steps = [
    ('Preprocessor', preprocessor),
    ('Model', optimized_model)
])

# Train the model
model_2.fit(x_train_2, y_train_2)

# Predict and evaluate
y_predict_2 = model_2.predict(x_test_2)
print(y_predict_2)
print("STROKE REPORT:")
print(classification_report(y_test_2, y_predict_2))